# Raman Spectroscopy Analysis - NAM Sample\n\n## Project Overview\n- **Instrument:** BWS465-785H (785 nm excitation)\n- **Detector:** BTC665N-785H-SYS\n- **Data Format:** 2048 spectral points per measurement\n- **X-axis:** Raman Shift (cm⁻¹)\n- **Y-axis:** Dark Subtracted Intensity (a.u.)\n\n---

In [1]:
# Cell 1: Imports and Setup\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom openpyxl import load_workbook\nimport os\nfrom pathlib import Path\n\n# Set base directory\nBASE_DIR = r\"c:\\Users\\sukes\\Downloads\\nam-new\"\nos.chdir(BASE_DIR)\n\n# Publication-quality plot settings\nplt.rcParams.update({\n    'font.size': 11,\n    'font.family': 'Arial',\n    'axes.linewidth': 1.2,\n    'xtick.major.width': 1.2,\n    'ytick.major.width': 1.2,\n    'figure.dpi': 150,\n    'savefig.dpi': 300,\n    'savefig.bbox': 'tight'\n})\n\nprint(\"Setup complete.\")\nprint(f\"Working directory: {os.getcwd()}\")

In [2]:
# Cell 2: Helper function to extract Raman spectrum from XLSX\n\ndef extract_spectrum(xlsx_path):\n    \"\"\"Extract Raman Shift (col 4) and Dark Subtracted (col 8) from XLSX.\"\"\"\n    wb = load_workbook(xlsx_path, read_only=True)\n    ws = wb.active\n    \n    raman_shift = []\n    intensity = []\n    \n    for row in ws.iter_rows(min_row=100, values_only=True):\n        try:\n            rs = float(row[3])   # Column 4: Raman Shift\n            ds = float(row[7])   # Column 8: Dark Subtracted #1\n            raman_shift.append(rs)\n            intensity.append(ds)\n        except (TypeError, ValueError, IndexError):\n            continue\n    \n    wb.close()\n    return np.array(raman_shift), np.array(intensity)\n\n\ndef load_folder_spectra(folder_path):\n    \"\"\"Load all XLSX spectra from a folder.\"\"\"\n    spectra = []\n    files = sorted([f for f in os.listdir(folder_path) if f.endswith('.xlsx')])\n    \n    for f in files:\n        rs, intensity = extract_spectrum(os.path.join(folder_path, f))\n        spectra.append({'file': f, 'raman_shift': rs, 'intensity': intensity})\n    \n    print(f\"Loaded {len(spectra)} spectra from: {folder_path}\")\n    return spectra\n\nprint(\"Helper functions loaded.\")

## Figure S1: Raw Spectra Comparison — Glass vs NAM\n\n**Purpose:** Demonstrate that glass substrate does not contribute Raman peaks.  \n**Glass:** `glass slide empty/empty slide/` (5 spectra)  \n**NAM:** `70p-60s-5ac-diifrentpoint/` (3 spectra, different spots)

In [3]:
# Cell 3: Load Glass and NAM spectra\n\n# Glass substrate (empty slide)\nglass_spectra = load_folder_spectra(r\"glass slide empty\\empty slide\")\n\n# NAM sample (70% power, 60s, 5 accumulations, different spots)\nnam_spectra = load_folder_spectra(r\"70p-60s-5ac-diifrentpoint\")

In [4]:
# Cell 4: Figure S1 — Glass vs NAM Raw Spectra\n\nfig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)\nfig.subplots_adjust(hspace=0.08)\n\n# Spectral range for display\nxmin, xmax = 200, 1800\n\n# --- Panel (a): Glass ---\nglass_colors = plt.cm.Blues(np.linspace(0.4, 0.9, 5))\nfor i, sp in enumerate(glass_spectra):\n    mask = (sp['raman_shift'] >= xmin) & (sp['raman_shift'] <= xmax)\n    ax1.plot(sp['raman_shift'][mask], sp['intensity'][mask],\n             linewidth=0.7, alpha=0.8, color=glass_colors[i],\n             label=f\"Glass {i+1}\")\n\nax1.set_ylabel('Intensity (a.u.)', fontsize=12)\nax1.set_title('(a) Empty Glass Slide — No Raman Peaks', fontsize=12, fontweight='bold', loc='left')\nax1.legend(loc='upper right', fontsize=9, framealpha=0.9)\nax1.tick_params(axis='both', labelsize=10)\nax1.set_xlim(xmin, xmax)\n\n# --- Panel (b): NAM ---\nnam_colors = ['#e41a1c', '#377eb8', '#4daf4a']\nfor i, sp in enumerate(nam_spectra):\n    mask = (sp['raman_shift'] >= xmin) & (sp['raman_shift'] <= xmax)\n    ax2.plot(sp['raman_shift'][mask], sp['intensity'][mask],\n             linewidth=0.7, alpha=0.85, color=nam_colors[i],\n             label=f\"NAM Spot {i+1}\")\n\nax2.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12)\nax2.set_ylabel('Intensity (a.u.)', fontsize=12)\nax2.set_title('(b) NAM Sample — Clear Raman Peaks', fontsize=12, fontweight='bold', loc='left')\nax2.legend(loc='upper right', fontsize=9, framealpha=0.9)\nax2.tick_params(axis='both', labelsize=10)\n\nplt.savefig(r'Analysis\\Figures\\Figure_S1_Glass_vs_NAM.png', dpi=300, facecolor='white')\nplt.savefig(r'Analysis\\Figures\\Figure_S1_Glass_vs_NAM.pdf', facecolor='white')\nplt.show()\nprint(\"Figure S1 saved.\")

## Reproducibility Check — Same Spot vs Different Spot\n\nCompare spectra taken at:\n- **Same spot** (repeatability): `90p-30s-5acc-samespot/`\n- **Different spots** (homogeneity): `90p-30s-5ac-diifrentpoint/`

In [5]:
# Cell 5: Load same-spot and different-spot data (90% power, 30s, 5 acc)\n\nsame_spot = load_folder_spectra(r\"90p-30s-5acc-samespot\")\ndiff_spot = load_folder_spectra(r\"90p-30s-5ac-diifrentpoint\")

In [6]:
# Cell 6: Figure 2 — Same Spot vs Different Spot Reproducibility\n\nfig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)\nfig.subplots_adjust(hspace=0.08)\n\nxmin, xmax = 200, 1800\n\n# --- Panel (a): Same Spot ---\nsame_colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(same_spot)))\nfor i, sp in enumerate(same_spot):\n    mask = (sp['raman_shift'] >= xmin) & (sp['raman_shift'] <= xmax)\n    ax1.plot(sp['raman_shift'][mask], sp['intensity'][mask],\n             linewidth=0.5, alpha=0.7, color=same_colors[i])\n\nax1.set_ylabel('Intensity (a.u.)', fontsize=12)\nax1.set_title(f'(a) Same Spot — {len(same_spot)} repeated measurements (90% power, 30s, 5 acc)',\n              fontsize=11, fontweight='bold', loc='left')\nax1.tick_params(axis='both', labelsize=10)\nax1.set_xlim(xmin, xmax)\n\n# --- Panel (b): Different Spots ---\ndiff_colors = plt.cm.Set1(np.linspace(0, 1, len(diff_spot)))\nfor i, sp in enumerate(diff_spot):\n    mask = (sp['raman_shift'] >= xmin) & (sp['raman_shift'] <= xmax)\n    ax2.plot(sp['raman_shift'][mask], sp['intensity'][mask],\n             linewidth=0.7, alpha=0.8, color=diff_colors[i],\n             label=f\"Spot {i+1}\")\n\nax2.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12)\nax2.set_ylabel('Intensity (a.u.)', fontsize=12)\nax2.set_title(f'(b) Different Spots — {len(diff_spot)} locations (90% power, 30s, 5 acc)',\n              fontsize=11, fontweight='bold', loc='left')\nax2.legend(loc='upper right', fontsize=9, framealpha=0.9)\nax2.tick_params(axis='both', labelsize=10)\n\nplt.savefig(r'Analysis\\Figures\\Figure_2_Reproducibility.png', dpi=300, facecolor='white')\nplt.savefig(r'Analysis\\Figures\\Figure_2_Reproducibility.pdf', facecolor='white')\nplt.show()\nprint(\"Figure 2 saved.\")

## Power & Time Comparison\n\nCompare different measurement conditions:\n- **70% power, 10s, 5 acc** (same spot): `NAM-70p-10s-5ac/`\n- **70% power, 10s, 3 acc** (same spot): `NAM-70p-10s-3acc/`\n- **90% power, 10s, 5 acc** (same spot): `NAM-90p-10s-5ac/`\n- **90% power, 10s, 3 acc** (same spot): `NAM-90p-10s-3ac/`\n- **90% power, 60s, 5 acc** (same spot): `90p-60s-5a-samespot/`\n- **70% power, 60s, 5 acc** (different spot): `70p-60s-5ac-diifrentpoint/`

In [7]:
# Cell 7: Load all condition datasets\n\nconditions = {\n    '70%, 10s, 5acc': load_folder_spectra(r\"NAM-70p-10s-5ac\"),\n    '70%, 10s, 3acc': load_folder_spectra(r\"NAM-70p-10s-3acc\"),\n    '90%, 10s, 5acc': load_folder_spectra(r\"NAM-90p-10s-5ac\"),\n    '90%, 10s, 3acc': load_folder_spectra(r\"NAM-90p-10s-3ac\"),\n    '90%, 60s, 5acc': load_folder_spectra(r\"90p-60s-5a-samespot\"),\n    '70%, 60s, 5acc': load_folder_spectra(r\"70p-60s-5ac-diifrentpoint\"),\n}\n\nprint(f\"\\nLoaded {len(conditions)} conditions.\")

In [8]:
# Cell 8: Figure 3 — Mean Spectrum per Condition (Overlay)\n\nfig, ax = plt.subplots(figsize=(12, 6))\n\nxmin, xmax = 200, 1800\ncolors = plt.cm.tab10(np.linspace(0, 1, len(conditions)))\noffset = 0  # vertical offset for stacking\n\nfor i, (label, spectra) in enumerate(conditions.items()):\n    # Compute mean spectrum\n    # Use the first spectrum's raman_shift as reference\n    rs = spectra[0]['raman_shift']\n    mask = (rs >= xmin) & (rs <= xmax)\n    rs_masked = rs[mask]\n    \n    intensities = np.array([sp['intensity'][mask] for sp in spectra])\n    mean_intensity = intensities.mean(axis=0)\n    \n    ax.plot(rs_masked, mean_intensity + offset, linewidth=1.0,\n            color=colors[i], label=f\"{label} (n={len(spectra)})\")\n    offset += mean_intensity.max() * 0.3  # offset for next trace\n\nax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12)\nax.set_ylabel('Intensity (a.u.) — stacked', fontsize=12)\nax.set_title('Mean Raman Spectrum per Measurement Condition', fontsize=13, fontweight='bold')\nax.legend(loc='upper right', fontsize=9, framealpha=0.9)\nax.set_xlim(xmin, xmax)\nax.tick_params(axis='both', labelsize=10)\n\nplt.savefig(r'Analysis\\Figures\\Figure_3_Conditions_Comparison.png', dpi=300, facecolor='white')\nplt.savefig(r'Analysis\\Figures\\Figure_3_Conditions_Comparison.pdf', facecolor='white')\nplt.show()\nprint(\"Figure 3 saved.\")

## Signal-to-Noise Ratio (SNR) Comparison\n\nCompare SNR across conditions to identify optimal measurement parameters.

In [9]:
# Cell 9: SNR analysis per condition\n\ndef compute_snr(raman_shift, intensity, signal_range=(900, 1100), noise_range=(1700, 1800)):\n    \"\"\"Compute SNR: max signal in signal_range / std of noise in noise_range.\"\"\"\n    sig_mask = (raman_shift >= signal_range[0]) & (raman_shift <= signal_range[1])\n    noise_mask = (raman_shift >= noise_range[0]) & (raman_shift <= noise_range[1])\n    \n    signal = np.max(intensity[sig_mask]) - np.median(intensity[noise_mask])\n    noise = np.std(intensity[noise_mask])\n    \n    return signal / noise if noise > 0 else 0\n\nprint(f\"{'Condition':<20} {'N spectra':<12} {'Mean SNR':<12} {'Std SNR':<12} {'Best SNR':<12}\")\nprint(\"-\" * 68)\n\nsnr_results = {}\nfor label, spectra in conditions.items():\n    snrs = []\n    for sp in spectra:\n        snr = compute_snr(sp['raman_shift'], sp['intensity'])\n        snrs.append(snr)\n    \n    snr_results[label] = snrs\n    print(f\"{label:<20} {len(spectra):<12} {np.mean(snrs):<12.1f} {np.std(snrs):<12.1f} {np.max(snrs):<12.1f}\")

In [10]:
# Cell 10: Figure 4 — SNR Bar Chart\n\nfig, ax = plt.subplots(figsize=(8, 5))\n\nlabels = list(snr_results.keys())\nmeans = [np.mean(v) for v in snr_results.values()]\nstds = [np.std(v) for v in snr_results.values()]\n\nbars = ax.bar(range(len(labels)), means, yerr=stds, capsize=5,\n              color=plt.cm.viridis(np.linspace(0.2, 0.8, len(labels))),\n              edgecolor='black', linewidth=0.8, alpha=0.85)\n\nax.set_xticks(range(len(labels)))\nax.set_xticklabels(labels, rotation=30, ha='right', fontsize=10)\nax.set_ylabel('Signal-to-Noise Ratio', fontsize=12)\nax.set_title('SNR Comparison Across Measurement Conditions', fontsize=13, fontweight='bold')\nax.grid(axis='y', alpha=0.3, linestyle='--')\n\n# Add value labels on bars\nfor bar, mean in zip(bars, means):\n    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,\n            f'{mean:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')\n\nplt.tight_layout()\nplt.savefig(r'Analysis\\Figures\\Figure_4_SNR_Comparison.png', dpi=300, facecolor='white')\nplt.savefig(r'Analysis\\Figures\\Figure_4_SNR_Comparison.pdf', facecolor='white')\nplt.show()\nprint(\"Figure 4 saved.\")

## Best Condition: Single High-Quality Spectrum\n\nPlot the best spectrum from the highest SNR condition for peak identification.

In [11]:
# Cell 11: Find best spectrum and plot with peak detection\nfrom scipy.signal import find_peaks\n\n# Find the condition + spectrum with highest SNR\nbest_condition = max(snr_results, key=lambda k: np.max(snr_results[k]))\nbest_idx = np.argmax(snr_results[best_condition])\nbest_spectrum = conditions[best_condition][best_idx]\n\nprint(f\"Best condition: {best_condition}\")\nprint(f\"Best spectrum: {best_spectrum['file']}\")\nprint(f\"SNR: {snr_results[best_condition][best_idx]:.1f}\")\n\n# Extract data\nrs = best_spectrum['raman_shift']\nintensity = best_spectrum['intensity']\n\n# Filter range\nxmin, xmax = 200, 1800\nmask = (rs >= xmin) & (rs <= xmax)\nrs_plot = rs[mask]\nint_plot = intensity[mask]\n\n# Find peaks\npeaks, properties = find_peaks(int_plot, height=np.median(int_plot) + 2*np.std(int_plot),\n                               distance=15, prominence=50)\n\nprint(f\"\\nPeaks detected: {len(peaks)}\")\nprint(f\"Peak positions (cm-1): {[f'{rs_plot[p]:.0f}' for p in peaks]}\")

In [12]:
# Cell 12: Figure 5 — Best Spectrum with Peak Labels\n\nfig, ax = plt.subplots(figsize=(12, 5))\n\nax.plot(rs_plot, int_plot, 'k-', linewidth=0.8, label=f'{best_condition}')\nax.plot(rs_plot[peaks], int_plot[peaks], 'rv', markersize=6, label='Detected peaks')\n\n# Label peaks\nfor p in peaks:\n    ax.annotate(f'{rs_plot[p]:.0f}',\n                xy=(rs_plot[p], int_plot[p]),\n                xytext=(0, 10), textcoords='offset points',\n                ha='center', va='bottom', fontsize=8, fontweight='bold',\n                color='red')\n\nax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12)\nax.set_ylabel('Intensity (a.u.)', fontsize=12)\nax.set_title(f'Best Spectrum — {best_condition} ({best_spectrum[\"file\"]})',\n             fontsize=13, fontweight='bold')\nax.legend(loc='upper right', fontsize=10)\nax.set_xlim(xmin, xmax)\nax.tick_params(axis='both', labelsize=10)\n\nplt.tight_layout()\nplt.savefig(r'Analysis\\Figures\\Figure_5_Best_Spectrum_Peaks.png', dpi=300, facecolor='white')\nplt.savefig(r'Analysis\\Figures\\Figure_5_Best_Spectrum_Peaks.pdf', facecolor='white')\nplt.show()\nprint(\"Figure 5 saved.\")

## All Folders Overview — Quick Visual Scan\n\nPlot one representative spectrum from every folder for a global view.

In [13]:
# Cell 13: Overview — One spectrum from each folder (stacked)\n\nall_folders = [\n    ('Glass (empty)', r'glass slide empty\\empty slide'),\n    ('70%, 10s, 5acc (same)', r'NAM-70p-10s-5ac'),\n    ('70%, 10s, 3acc (same)', r'NAM-70p-10s-3acc'),\n    ('70%, 10s, 5acc (diff)', r'70p-5ac-10s-diifrentpoint'),\n    ('70%, 30s, 5acc (diff)', r'70p-5ac-30s-diifrentspot'),\n    ('70%, 60s, 5acc (diff)', r'70p-60s-5ac-diifrentpoint'),\n    ('90%, 10s, 5acc (same)', r'NAM-90p-10s-5ac'),\n    ('90%, 10s, 3acc (same)', r'NAM-90p-10s-3ac'),\n    ('90%, 30s, 5acc (same)', r'90p-30s-5acc-samespot'),\n    ('90%, 30s, 5acc (diff)', r'90p-30s-5ac-diifrentpoint'),\n    ('90%, 60s, 5acc (same)', r'90p-60s-5a-samespot'),\n    ('90%, 60s, 5acc (diff)', r'90p-60s-5ac-diffrentspot'),\n]\n\nfig, ax = plt.subplots(figsize=(12, 10))\n\nxmin, xmax = 200, 1800\noffset = 0\ncolors = plt.cm.tab20(np.linspace(0, 1, len(all_folders)))\n\nfor i, (label, folder) in enumerate(all_folders):\n    # Get first XLSX file\n    files = sorted([f for f in os.listdir(folder) if f.endswith('.xlsx')])\n    if not files:\n        continue\n    rs, intensity = extract_spectrum(os.path.join(folder, files[0]))\n    mask = (rs >= xmin) & (rs <= xmax)\n    \n    ax.plot(rs[mask], intensity[mask] + offset, linewidth=0.7,\n            color=colors[i], label=label)\n    offset += np.max(intensity[mask]) * 0.4\n\nax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=12)\nax.set_ylabel('Intensity (a.u.) — stacked', fontsize=12)\nax.set_title('All Measurement Conditions — First Spectrum per Folder', fontsize=13, fontweight='bold')\nax.legend(loc='upper right', fontsize=8, framealpha=0.9, ncol=2)\nax.set_xlim(xmin, xmax)\n\nplt.tight_layout()\nplt.savefig(r'Analysis\\Figures\\Figure_Overview_All_Folders.png', dpi=300, facecolor='white')\nplt.savefig(r'Analysis\\Figures\\Figure_Overview_All_Folders.pdf', facecolor='white')\nplt.show()\nprint(\"Overview figure saved.\")